## Phase 3 - Preprocessing

This section transforms the raw DDI dataset into a clean, structured, and model-ready format. The preprocessing pipeline is designed to standardise text, reduce noise, and extract meaningful features while preserving critical biomedical information.

The workflow begins with drug name normalisation to ensure consistency across different representations. Drug entities are then masked using generic tokens (`DRUG1`, `DRUG2`) to prevent models from memorising specific drug names and instead focus on contextual interaction patterns.

The masked text is cleaned and processed using scispaCy for tokenisation, stop-word removal (with negation preserved), and lemmatisation. In addition, domain-informed keyword features are extracted to capture pharmacological signals such as mechanism, effect, and advisory interactions.

The result is a comprehensive dataset containing multiple textual representations and engineered features, enabling robust experimentation with both classical machine learning and transformer-based models.

## Setup and Library Imports

This section initialises the environment and imports all required libraries for preprocessing. The project configuration is loaded to ensure consistent access to file paths, parameters, and settings across notebooks.

Core libraries such as `pandas` are used for data handling, while `re` supports text cleaning operations. The `spacy` library is introduced for advanced NLP preprocessing tasks such as tokenisation and lemmatisation. Progress tracking is enabled using `tqdm` for efficient monitoring of longer operations.

This setup ensures a consistent and reproducible preprocessing pipeline.

In [1]:
import sys, os
# Adjust depth so Python can find config.py at the project root
sys.path.insert(0, os.path.abspath('../'))
from config import *

import re, warnings
import pandas as pd
import spacy
from tqdm import tqdm

warnings.filterwarnings('ignore')

## Data Path Verification

This step verifies the availability of input datasets required for preprocessing. The training and test CSV files, generated in Phase 1, are checked to ensure they exist at the expected locations defined in the configuration file.

Validating file paths before processing helps prevent runtime errors and ensures that the preprocessing pipeline operates on the correct data. This is a simple but essential step for maintaining reproducibility and robustness in the workflow.

In [2]:
# Paths come from config.py
for path in [TRAIN_CSV, TEST_CSV]:
    status = 'FOUND' if os.path.exists(path) else 'NOT FOUND'

## Initialising scispaCy Model

This step loads the scispaCy language model (`en_core_sci_sm`) for domain-specific text processing.

scispaCy is designed for biomedical and scientific text, providing improved tokenisation and linguistic handling compared to standard spaCy models. Named entity recognition and dependency parsing are disabled to optimise performance, as only lightweight preprocessing is required at this stage.

The maximum document length is increased to handle longer biomedical sentences without runtime errors.

In [3]:
%pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

  Using cached https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz (14.8 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
print('Loading scispaCy model')

try:

    nlp = spacy.load(
        'en_core_sci_sm',
        disable=['ner', 'parser']
    )

    nlp.max_length = 2_000_000

    print('en_core_sci_sm loaded successfully.')

    print(
        f'Active pipeline components: {nlp.pipe_names}'
    )

except OSError:

    raise OSError(
        "scispaCy model 'en_core_sci_sm' not found.\n"
        "See README or requirements.txt for installation instructions."
    ) from None

Loading scispaCy model
en_core_sci_sm loaded successfully.
Active pipeline components: ['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer']


## Custom Stop Words with Negation Preservation

This step refines the default stop word list by explicitly preserving negation words such as “not”, “no”, and “without”. In biomedical text, negation plays a critical role in determining whether an interaction exists or is explicitly denied.

Removing negation words during preprocessing would lead to loss of important semantic information and could negatively impact model performance. Therefore, these words are excluded from the stop word list to ensure they are retained during text processing.

This modification ensures that the preprocessing pipeline maintains meaningful linguistic cues, particularly for distinguishing between interaction and non-interaction cases.

In [5]:
# These words must be preserved they flip the meaning of a sentence
NEGATION_WORDS = {'not', 'no', 'without', 'neither', 'nor', 'never', 'none'}

# Remove negation words from the standard stop word list
CUSTOM_STOP_WORDS = nlp.Defaults.stop_words - NEGATION_WORDS

print(f'Standard stop words : {len(nlp.Defaults.stop_words)}')
print(f'After removing negation words: {len(CUSTOM_STOP_WORDS)}')
print(f'Preserved negation words : {sorted(NEGATION_WORDS)}')

# Verify negation words are NOT in our custom list
check = [w for w in NEGATION_WORDS if w in CUSTOM_STOP_WORDS]
print(f'Negation words still in stop list (should be empty): {check}')

Standard stop words : 326
After removing negation words: 319
Preserved negation words : ['neither', 'never', 'no', 'none', 'nor', 'not', 'without']
Negation words still in stop list (should be empty): []


## Drug Name Normalisation

This step standardises drug names to ensure consistency across the dataset. Drug mentions often appear in different formats (e.g., uppercase, lowercase, spacing variations, or textual representations of Greek letters), which can introduce noise and reduce matching accuracy.

The normalisation process converts all names to lowercase, replaces Greek letter words (e.g., “beta”, “gamma”) with their symbolic forms, and standardises formatting such as hyphen spacing and whitespace. This ensures that semantically identical drug names are treated consistently.

A small set of test cases is included to validate the correctness of the transformation, ensuring robustness before applying it across the dataset.

In [6]:
GREEK_MAP = {
    'alpha': 'α', 'beta': 'β', 'gamma': 'γ', 'delta': 'δ',
    'epsilon': 'ε', 'omega': 'ω', 'mu': 'μ',
}

def normalise_drug_name(name: str) -> str:
    """
    Normalise a drug name to a canonical lowercase form.
    """
    if not name or not isinstance(name, str):
        return ''

    name = name.strip().lower()

    # Replace Greek letter word spellings with actual symbols
    for word, symbol in GREEK_MAP.items():
        name = re.sub(rf'\b{word}\b', symbol, name)

    # Normalise spaces around hyphens
    name = re.sub(r'\s*-\s*', '-', name)

    # Collapse multiple whitespace
    name = re.sub(r'\s+', ' ', name).strip()

    return name


# Quick tests
test_cases = [
    ('Warfarin', 'warfarin'),
    ('ASPIRIN', 'aspirin'),
    ('beta-blocker', 'β-blocker'),
    ('anti - coagulant', 'anti-coagulant'),
    ('gamma-aminobutyric acid','γ-aminobutyric acid')
    
]

print('Drug name normalisation tests:')

all_pass = True
for inp, expected in test_cases:
    result = normalise_drug_name(inp)
    status = 'True' if result == expected else f'FAIL (got: {result})'
    print(f'{repr(inp):<30} gives {repr(result):<35} {status}')
    if status != 'True':
        all_pass = False

print(f'\nAll tests passed: {all_pass}')

Drug name normalisation tests:
'Warfarin'                     gives 'warfarin'                          True
'ASPIRIN'                      gives 'aspirin'                           True
'beta-blocker'                 gives 'β-blocker'                         True
'anti - coagulant'             gives 'anti-coagulant'                    True
'gamma-aminobutyric acid'      gives 'γ-aminobutyric acid'               True

All tests passed: True


## Drug Entity Masking

This step replaces explicit drug names in each sentence with generic tokens (`DRUG1`, `DRUG2`) to prevent models from memorising specific drug identities. This encourages the model to focus on contextual patterns rather than overfitting to frequent drug names.

All occurrences of each drug are replaced, ensuring consistency even when a drug appears multiple times in a sentence. To avoid incorrect partial replacements, longer drug names are processed first. Case-insensitive matching is used to handle variations in text.

The chosen token format ensures compatibility with downstream cleaning steps, preserving these markers throughout preprocessing. Verification tests confirm correct masking behaviour and robustness of the implementation.

In [7]:
def mask_drug_entities(sentence: str, drug1: str, drug2: str) -> str:
    """
    Replace all occurrences of drug1 and drug2 with DRUG1 / DRUG2 tokens.
    """
    masked = sentence

    # Replace longer name first to avoid partial matches
    pairs = sorted(
        [(drug1, 'DRUG1'), (drug2, 'DRUG2')],
        key=lambda x: len(x[0]),
        reverse=True
    )

    for drug, token in pairs:
        if not drug or not isinstance(drug, str):
            continue
        pattern = re.compile(re.escape(drug.strip()), re.IGNORECASE)
        masked = pattern.sub(token, masked)

    return masked


# Verification tests
print('Masking tests:')

# Test 1: basic masking
t1 = mask_drug_entities(
    'Warfarin increases bleeding risk when combined with aspirin.',
    'Warfarin', 'aspirin'
)
print(f'Test 1 (basic): {t1}')

# Test 2: multiple mentions
t2 = mask_drug_entities(
    'Aspirin may interact with aspirin in high doses combined with warfarin.',
    'Aspirin', 'warfarin'
)
print(f'Test 2 (multi): {t2}')
assert 'Aspirin' not in t2

# Test 3: tokens survive cleaning
cleaned = t1.lower()
cleaned = re.sub(r'[^a-z0-9\-\'\s]', ' ', cleaned)
cleaned = re.sub(r'\s+', ' ', cleaned).strip()
print(f'Test 3 (survives clean): {cleaned}')
assert 'drug1' in cleaned and 'drug2' in cleaned

Masking tests:
Test 1 (basic): DRUG1 increases bleeding risk when combined with DRUG2.
Test 2 (multi): DRUG1 may interact with DRUG1 in high doses combined with DRUG2.
Test 3 (survives clean): drug1 increases bleeding risk when combined with drug2


## Text Cleaning

This step standardises sentence text to a consistent format suitable for downstream NLP processing. The goal is to reduce noise while preserving medically relevant structure.

The cleaning pipeline performs three key operations: converting text to lowercase, removing non-essential special characters, and normalising whitespace. Importantly, hyphens and apostrophes are retained, as they are common in biomedical terminology (e.g., *anti-coagulant*, *drug-induced*).

Drug masking must be applied before this step, as the `DRUG1` and `DRUG2` tokens are designed to survive this cleaning process. This ensures that entity placeholders remain intact for subsequent modelling.

The output is a clean, normalised text representation that balances simplicity with preservation of domain-specific information.

In [8]:
def clean_text(text: str) -> str:
    """
    Standardise text for NLP processing.
    """
    if not isinstance(text, str):
        return ''

    text = text.lower()

    # Keep alphanumerics, hyphens, apostrophes, spaces
    text = re.sub(r"[^a-z0-9\-\'\s]", ' ', text)

    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# Tests
print('Cleaning tests:')

tests = [
    'DRUG1 increases (bleeding) risk with DRUG2.',
    'anti-coagulant therapy; contraindicated!'
]

for t in tests:
    print(f'In:  {t}')
    print(f'Out: {clean_text(t)}')
    print()

Cleaning tests:
In:  DRUG1 increases (bleeding) risk with DRUG2.
Out: drug1 increases bleeding risk with drug2

In:  anti-coagulant therapy; contraindicated!
Out: anti-coagulant therapy contraindicated



## Keyword-Based Feature Extraction

This step extracts domain-informed lexical signals from each sentence to support downstream modelling. Based on insights from Phase 2 EDA, keywords are grouped into interaction-specific categories: *mechanism*, *effect*, *advise*, and *general*.

A prefix-based matching strategy is used, allowing the capture of multiple morphological variants (e.g., *inhibit*, *inhibits*, *inhibition*) while avoiding incorrect matches (e.g., *uninhibited*). This balances recall and precision without requiring full stemming or lemmatisation.

The output consists of both aggregated and type-specific keyword counts, along with the list of detected keywords. These features provide interpretable signals that complement statistical and transformer-based representations.

Notably, overly generic terms such as *effect* are excluded to reduce noise and prevent misleading feature activation.

In [9]:
KEYWORDS_BY_TYPE = {
    'mechanism': [
        'inhibit', 'induce', 'metabolis', 'metaboliz', 'clearance',
        'absorption', 'bioavailability', 'cyp', 'enzyme', 'excret',
        'bind', 'plasma', 'concentration', 'half-life',
    ],
    'effect': [
        'bleeding', 'toxicity', 'adverse', 'sedation', 'hypotension',
        'bradycardia', 'elevat', 'risk', 'potentiat', 'enhance',
    ],
    'advise': [
        'contraindicated', 'avoid', 'caution', 'monitor', 'recommend',
        'warning', 'concomitant', 'concurrent',
    ],
    'general': [
        'increase', 'decrease', 'reduc', 'lower', 'block',
        'interact', 'dose', 'level', 'activity',
    ],
}

ALL_KEYWORDS = [kw for kws in KEYWORDS_BY_TYPE.values() for kw in kws]

KEYWORD_PATTERNS = {
    kw: re.compile(rf'\b{re.escape(kw)}', re.IGNORECASE)
    for kw in ALL_KEYWORDS
}


def extract_keywords(cleaned_text: str) -> dict:
    found = []
    type_counts = {t: 0 for t in KEYWORDS_BY_TYPE}

    for kw_type, keywords in KEYWORDS_BY_TYPE.items():
        for kw in keywords:
            if KEYWORD_PATTERNS[kw].search(cleaned_text):
                found.append(kw)
                type_counts[kw_type] += 1

    return {
        'keywords_found': ', '.join(found) if found else '',
        'keyword_count': len(found),
        'kw_mechanism': type_counts['mechanism'],
        'kw_effect': type_counts['effect'],
        'kw_advise': type_counts['advise'],
        'kw_general': type_counts['general'],
    }


# Tests
print('Keyword extraction tests:')

t1 = extract_keywords('The patient showed increased risk with decreased doses')
print(f'Test 1: {t1["keywords_found"]}')

t2 = extract_keywords('warfarin inhibits cyp metabolism increasing toxicity risk')
print(f'Test 2: {t2["keywords_found"]}')


Keyword extraction tests:
Test 1: risk, increase, decrease, dose
Test 2: inhibit, metabolis, cyp, toxicity, risk


## End-to-End Sentence Preprocessing Pipeline

This function integrates all preprocessing steps into a single, consistent pipeline applied to each drug–drug pair sentence. It ensures that all transformations are performed in the correct order and produces multiple representations of the input for downstream modelling.

The pipeline begins by identifying *self-pairs*, where both drug mentions refer to the same textual entity. It then applies entity masking to replace drug names with generic tokens, followed by text cleaning to standardise the input.

Next, the cleaned sentence is processed using scispaCy to obtain tokenised and lemmatised representations, with stop words removed while preserving negation terms. Finally, keyword-based features are extracted to capture domain-specific interaction signals.

The function returns a structured dictionary containing all intermediate and final representations, allowing flexible use across different modelling approaches (e.g., classical ML, feature-based models, and transformers).

In [10]:
def preprocess_sentence(sentence: str, drug1: str, drug2: str) -> dict:
    # Step 1: detect self-pairs
    is_self_pair = (
        str(drug1).strip().lower() == str(drug2).strip().lower()
    )

    # Step 2: mask drug names
    masked = mask_drug_entities(str(sentence), str(drug1), str(drug2))

    # Step 3: clean text
    cleaned = clean_text(masked)

    # Step 4: tokenisation + stop word removal + lemmatisation
    doc = nlp(cleaned)
    tokens_all = [t.text for t in doc]
    tokens_no_stop = [t.text for t in doc if t.text not in CUSTOM_STOP_WORDS]
    lemmas = [t.lemma_ for t in doc if t.text not in CUSTOM_STOP_WORDS]

    # Step 5: keyword extraction
    kw_result = extract_keywords(cleaned)

    return {
        'sentence_original': sentence,
        'sentence_masked': masked,
        'sentence_cleaned': cleaned,
        'tokens': ' '.join(tokens_all),
        'tokens_no_stop': ' '.join(tokens_no_stop),
        'lemmas': ' '.join(lemmas),
        'is_self_pair': is_self_pair,
        **kw_result,
    }

## Full Dataset Preprocessing Pipeline

This function applies the complete preprocessing workflow to an entire dataset (train or test split) and produces a structured, model-ready output.

The pipeline begins by loading the raw CSV file and normalising drug names to ensure consistency. Each sentence is then processed using the previously defined `preprocess_sentence()` function, which performs masking, cleaning, tokenisation, lemmatisation, and keyword extraction.

To improve efficiency, the implementation uses `itertuples()` instead of `iterrows()`, significantly reducing runtime for large datasets. A progress bar is included for transparency during execution.

The processed outputs are merged with essential metadata fields such as drug types, corpus source, and interaction labels. This ensures that no useful contextual information is lost and enables richer feature engineering in later phases.

The final dataset is saved as a CSV file, containing both original attributes and newly derived features, forming the foundation for modelling and evaluation in subsequent stages.

In [11]:
def run_pipeline(input_path: str, output_path: str) -> pd.DataFrame:
    df = pd.read_csv(input_path)
    print(f'{len(df):,} rows loaded')

    # Normalise drug names
    df['drug1_norm'] = df['drug1_text'].astype(str).apply(normalise_drug_name)
    df['drug2_norm'] = df['drug2_text'].astype(str).apply(normalise_drug_name)

    # Process each row
    processed_rows = []

    for row in tqdm(df.itertuples(index=False), total=len(df), desc='Processing'):
        result = preprocess_sentence(
            sentence=str(row.sentence_text),
            drug1=str(row.drug1_text),
            drug2=str(row.drug2_text),
        )
        processed_rows.append(result)

    processed_df = pd.DataFrame(processed_rows)

    # Merge with metadata
    keep_cols = [
        'sentence_id',
        'drug1_text', 'drug1_type', 'drug1_norm',
        'drug2_text', 'drug2_type', 'drug2_norm',
        'ddi', 'interaction_type',
        'corpus_source',
        'offset_valid',
        'source_file',
    ]

    keep_cols = [c for c in keep_cols if c in df.columns]

    final_df = pd.concat(
        [df[keep_cols].reset_index(drop=True), processed_df],
        axis=1
    )

    # Save output
    os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)
    final_df.to_csv(output_path, index=False)

    return final_df

## Processing Training Data

In this step, the full preprocessing pipeline is applied to the training dataset. This transforms the raw input data into a structured, feature-rich format suitable for model development.

All preprocessing steps including drug name normalisation, entity masking, text cleaning, tokenisation, lemmatisation, and keyword extraction are executed for every row in the training set.

The resulting processed dataset is saved to the configured output path and will be used in subsequent phases for feature engineering and model training.

In [12]:
print('-' * 28)
print('PROCESSING TRAINING DATA')
print('-' * 28)

train_df = run_pipeline(TRAIN_CSV, TRAIN_PROCESSED)

----------------------------
PROCESSING TRAINING DATA
----------------------------
27,792 rows loaded


Processing: 100%|██████████| 27792/27792 [01:26<00:00, 321.05it/s]


## Processing Test Data

This step applies the same preprocessing pipeline to the test dataset, ensuring consistency between training and evaluation data.

Maintaining identical preprocessing for both splits is critical to avoid data leakage and ensure that model performance reflects true generalisation. All transformations including masking, cleaning, tokenisation, and feature extraction are applied in exactly the same manner as in the training phase.

The processed test dataset is saved and will be used for final evaluation and benchmarking in later stages of the project.

In [13]:
print('-' * 28)
print('PROCESSING TEST DATA')
print('-' * 28)

test_df = run_pipeline(TEST_CSV, TEST_PROCESSED)

----------------------------
PROCESSING TEST DATA
----------------------------
6,657 rows loaded


Processing: 100%|██████████| 6657/6657 [00:18<00:00, 354.29it/s]


## Output Verification and Sample Inspection

This step performs a sanity check on the processed training dataset to ensure that all preprocessing stages have been applied correctly.

First, the output schema is inspected to confirm that all expected columns including cleaned text, tokenised forms, and engineered features are present. 

Next, a sample positive interaction is examined across different representations (original, masked, cleaned, lemmatised, and keyword features) to verify that transformations preserve semantic meaning while standardising structure.

Finally, self-pair cases are inspected to validate correct identification and masking behaviour. These examples help confirm that edge cases are handled appropriately.

This verification step is essential before proceeding to feature engineering and modelling, ensuring data integrity and correctness of the preprocessing pipeline.

In [14]:
print('--- Output columns: ---')
print(list(train_df.columns))
print()

print('--- Positive interaction example ---')
pos = train_df[train_df['ddi'] == True].iloc[0]

for col in [
    'sentence_original',
    'sentence_masked',
    'sentence_cleaned',
    'lemmas',
    'keywords_found',
    'interaction_type'
]:
    val = str(pos[col])
    print(f'{col:<22}: {val[:90]}')

print()

print('--- Self-pair example ---')
self_pairs = train_df[train_df['is_self_pair'] == True]

if len(self_pairs) > 0:
    sp = self_pairs.iloc[0]
    print(f'drug1: {sp["drug1_text"]}  drug2: {sp["drug2_text"]}')
    print(f'masked: {sp["sentence_masked"][:80]}')
    print(f'label: {sp["interaction_type"]}')

print(f'\nTotal self-pairs in training set: {len(self_pairs)}')

--- Output columns: ---
['sentence_id', 'drug1_text', 'drug1_type', 'drug1_norm', 'drug2_text', 'drug2_type', 'drug2_norm', 'ddi', 'interaction_type', 'corpus_source', 'offset_valid', 'source_file', 'sentence_original', 'sentence_masked', 'sentence_cleaned', 'tokens', 'tokens_no_stop', 'lemmas', 'is_self_pair', 'keywords_found', 'keyword_count', 'kw_mechanism', 'kw_effect', 'kw_advise', 'kw_general']

--- Positive interaction example ---
sentence_original     : Concurrent administration of a TNF antagonist with ORENCIA has been associated with an inc
sentence_masked       : Concurrent administration of a DRUG1 with DRUG2 has been associated with an increased risk
sentence_cleaned      : concurrent administration of a drug1 with drug2 has been associated with an increased risk
lemmas                : concurrent administration drug1 drug2 associate increase risk infection no significant add
keywords_found        : risk, concurrent, increase
interaction_type      : effect

--- Self-pair e

## Pipeline Validation and Feature Sanity Checks

This step validates critical preprocessing fixes and inspects the quality of engineered features.

First, it verifies that the `DRUG1` and `DRUG2` masking tokens are correctly preserved after cleaning. This is essential, as earlier implementations using special characters (e.g., `@DRUG1$`) were removed during text cleaning, leading to loss of entity information.

The absence of legacy broken tokens confirms that the masking strategy is now robust and compatible with the cleaning pipeline.

Next, keyword-based features are analysed by computing their average counts across interaction types. This provides a quick sanity check that domain-informed keywords align with expected classes (e.g., mechanism-related keywords appearing more frequently in *mechanism* interactions).

Together, these checks ensure that both structural transformations and feature engineering steps are functioning as intended before moving to modelling.

In [15]:
# Verify DRUG1/DRUG2 tokens survived cleaning

has_drug1 = train_df['sentence_masked'].str.contains('DRUG1').sum()
has_drug2 = train_df['sentence_masked'].str.contains('DRUG2').sum()
total = len(train_df)

print(f'Rows with DRUG1 token : {has_drug1:,} / {total:,}')
print(f'Rows with DRUG2 token : {has_drug2:,} / {total:,}')

# Check old broken tokens are gone
broken = train_df['sentence_masked'].str.contains('@DRUG').sum()
print(f'Rows with broken @DRUG token: {broken} (should be 0)')

print()

# Keyword feature inspection
print('--- Keyword extraction sample ---')

kw_cols = ['keyword_count','kw_mechanism','kw_effect','kw_advise','kw_general']

print(
    train_df
    .groupby('interaction_type')[kw_cols]
    .mean()
    .round(2)
    .to_string()
)

Rows with DRUG1 token : 27,720 / 27,792
Rows with DRUG2 token : 25,731 / 27,792
Rows with broken @DRUG token: 0 (should be 0)

--- Keyword extraction sample ---
                  keyword_count  kw_mechanism  kw_effect  kw_advise  kw_general
interaction_type                                                               
advise                     2.07          0.48       0.14       0.86        0.59
effect                     1.55          0.27       0.40       0.26        0.63
false                      2.27          0.96       0.14       0.23        0.95
int                        1.71          0.41       0.06       0.17        1.07
mechanism                  3.34          1.72       0.12       0.21        1.29


## Data Quality Validation

This section performs a series of integrity checks to ensure that the preprocessing pipeline has produced a clean, consistent, and model-ready dataset.

The validation covers several critical aspects:

- **Missing Values**: Confirms that essential columns (cleaned text, lemmas, labels) contain no null entries.
- **Entity Masking Integrity**: Ensures that `DRUG1` and `DRUG2` tokens are correctly present in all non-self-pair sentences.
- **Label Consistency**: Verifies that all interaction labels belong to the predefined valid set, preventing downstream training errors.
- **Lemmatization Coverage**: Checks that tokenisation and lemmatisation have produced non-empty outputs.
- **Self-Pair Tracking**: Confirms correct identification of sentences where both drug mentions refer to the same textual entity.
- **Class Distribution**: Provides an overview of dataset imbalance, reinforcing the need for metrics such as Macro F1 and class-weighted training.

These checks act as a final safeguard before moving to feature engineering and modelling, ensuring that the dataset is both structurally sound and semantically reliable.

In [22]:
print('--- Quality checks ---\n')

passed = True

# Check 1: No nulls in critical columns
critical = ['sentence_masked','sentence_cleaned','lemmas','interaction_type']
nulls = train_df[critical].isnull().sum()

print('1. Null values in critical columns:')
print(nulls.to_string())

print('' if nulls.sum() == 0 else 'FAIL')
if nulls.sum() > 0:
    passed = False

# Check 2: DRUG1/DRUG2 tokens present
no_token = (~train_df['sentence_masked'].str.contains('DRUG1') |
            ~train_df['sentence_masked'].str.contains('DRUG2')).sum()

print(f'\n2. Rows missing DRUG1 or DRUG2 token: {no_token}')

no_token_excl = no_token - len(train_df[train_df['is_self_pair']==True])
print(f'(Excluding self-pairs: {max(0,no_token_excl)})')

print('' if no_token_excl <= 0 else 'some non-self-pair rows missing tokens')

# Check 3: Valid labels only
valid = {'false', 'effect', 'mechanism', 'advise', 'int'}
unexpected = set(train_df['interaction_type'].unique()) - valid

print(f'\n3. Unexpected labels: {unexpected if unexpected else "none"}')
print('' if not unexpected else 'FAIL')

if unexpected:
    passed = False

# Check 4: Lemmas column not empty
empty_lemmas = (train_df['lemmas'].str.strip() == '').sum()

print(f'\n4. Rows with empty lemmas: {empty_lemmas}')
print('' if empty_lemmas == 0 else 'WARN')

# Check 5: Self-pair count
self_count = train_df['is_self_pair'].sum()

print(f'\n5. Self-pairs flagged: {self_count} ({self_count/len(train_df)*100:.1f}%)')
print('These are now tracked and can be filtered if needed')

# Check 6: Class distribution
print('\n6. Class distribution:')

counts = train_df['interaction_type'].value_counts()

for label, count in counts.items():
    pct = count / len(train_df) * 100
    bar = chr(9608) * int(pct / 2)
    print(f'{label:<15} {count:6,}  ({pct:5.1f}%)')

--- Quality checks ---

1. Null values in critical columns:
sentence_masked     0
sentence_cleaned    0
lemmas              0
interaction_type    0


2. Rows missing DRUG1 or DRUG2 token: 2123
(Excluding self-pairs: 133)
some non-self-pair rows missing tokens

3. Unexpected labels: none


4. Rows with empty lemmas: 0


5. Self-pairs flagged: 1990 (7.2%)
These are now tracked and can be filtered if needed

6. Class distribution:
false           23,771  ( 85.5%)
effect           1,687  (  6.1%)
mechanism        1,319  (  4.7%)
advise             826  (  3.0%)
int                189  (  0.7%)


## Summary

The preprocessing pipeline successfully converts raw biomedical text into a structured and feature-rich dataset suitable for modelling. Key improvements include consistent drug name normalisation, robust entity masking, and preservation of important linguistic signals such as negation.

Multiple representations of each sentence are generated, including cleaned text, tokenised forms, and lemmatised sequences. In addition, keyword-based features provide interpretable signals aligned with interaction types, complementing statistical and deep learning approaches.

Quality checks confirm that the dataset is free from critical issues such as missing values, invalid labels, and broken masking tokens. Class imbalance remains significant, reinforcing the need for appropriate evaluation metrics such as Macro F1 in later stages.

Overall, the preprocessing stage establishes a reliable foundation for feature engineering, model training, and evaluation in subsequent phases of the project.